# Generalization

<div style="color: red;">
Hi! I'm Mike. Welcome to this Generalization code review tutorial. 

Once we train a model, what do we wanna do with it? We wanna apply it to unseen data and we wanna see how well it performs. This is so-called generalization.

So how can we make our model perform as well as possible on unseen data? How can we improve its cability to generalize? That is what I'm gonna discuss in today's tutorial.
</div>

<div style="color: red;">
Here is an outline of today's tutorial:
- In the first section, we are gonna look at what is generalization.
- Next, I'm gonna introduce why do we need to care about generalization.
- Then, we are gonna learn how can we select a model that generalizes well.
- Finally, we are gonna discuss how can we evaluate the final model.
</div>

<div style="color: red;">So, let's begin with the first section.</div>

## 1. What Is Generalization?

### 1.1 IID Assumption

- **Independent:** Samples are independent of one another.
- **Identically distributed:** Samples are drawn from the same distribution $P$.
- Training, validation, test sets are all assumed to be sampled from $P$.

### 1.2 What Is Generalization?

> Generalization is a model's capability to perform well on unseen samples sampled from the same underlying data distribution.

### 1.3 Generalization Error

Generalization error is the expected loss of the trained model on samples drawn from $P$.

<div style="color: red;">
Now assume we draw a sample `(x, y)` from the distribution `P`. We pass `x` through the model `f` to obtain the prediction `f(x)`, compare that prediction with the true value `y`, and compute the loss. The expected loss over samples from distribution `P` is called the generalization error.
</div>

$$
Err(f) = \mathbb{E}_{x,y \sim p}[\ell(f(x), y)].
$$

However, the true distribution $P$ is unknown. How can we calculate this expectation in practice?

We can use an independent test set to estimate the generalization error:

<div style="color: red;">
The true distribution `P` is usually unknown, so we can't actually calculate this expectation directly. 

So how can we do that? We can approximate it using a test set that was not involved in training the model. Since the test set is finite, we can compute the loss for each test sample and take the average of them. 

This quantity is called the test error, or test loss, and it provides an estimate of the generalization error.
</div>

$$
\widehat{Err}(f) = \mathbb{E}_{x, y \sim \mathcal{D}_{\text{test}}}[\ell(f(x), y)] = \frac{1}{|\mathcal{D}_{\text{test}}|} \sum_{x,y \in \mathcal{D}_{\text{test}}} \ell(f(x), y).
$$

$$
\text{test error} \approx \text{generalization error}
$$

## 2. Why Do We Care about Generalization?

Before answering this question, let's look at a KNN example.

<div style="color: red;">
First, let's import the libraries we need.
</div>

In [ ]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

from sklearn import datasets, neighbors

# Make the random splits reproducible
np.random.seed(1234)

<div style="color: red;">
Next, we need to load the California housing dataset. This is a regression dataset for predicting house values. The input features include information such as population and the number of rooms, and the target `y` is the house value.
<br><br>
We can then print the dimensions of the dataset.
<br><br>
We need to define a loss function as well. Here, we use mean squared error, which is a common loss function for regression tasks. 

The function has two arguments. `y` is the true values, and `y_hat` is the model's predictions. We subtract the predictions from the true values, then square the differences, and then take the average. 
</div>

In [ ]:
# Load the California housing regression dataset
x, y = datasets.fetch_california_housing(return_X_y=True)

num_instances, num_features = x.shape
print(num_instances, num_features)

# Mean squared error
def loss(y, yh):
    return np.mean((y - yh) ** 2)

<div style="color: red;">
Then, we use 20% of the data as an unseen test set.

Next, we generate a random permutation of the sample indices to shuffle the data. This is a common technique that prevents the train-test split from being influenced by the original ordering of the dataset.

We use the first `n_test` shuffled samples as the test set and the remaining samples as the training set. Once the two datasets are ready, we can start training the KNN model.
</div>

In [ ]:
# Use 20% of the data as an unseen test set
n_test = num_instances // 5

# Shuffle the sample indices before splitting
inds = np.random.permutation(num_instances)

x_test, y_test = x[inds[:n_test]], y[inds[:n_test]]
x_train, y_train = x[inds[n_test:]], y[inds[n_test:]]

<div style="color: red;">
Now suppose we want to evaluate KNN using different numbers of neighbours. How can we do that?
<br><br>
First, we create a range of candidate values of K from 1 to 100 and loop over them. At each iteration, we create a KNN model with the current value of `K`.
<br><br>
We use the `fit` method to train the KNN model on the training set. We then use `predict` to evaluate the model, compute both the training error and the test error, and store them in the `err_train` and `err_test` lists.
</div>

In [ ]:
K_list = range(1, 100)
err_train, err_test = [], []

for K in K_list:
    model = neighbors.KNeighborsRegressor(n_neighbors=K)
    model.fit(x_train, y_train)

    err_train.append(loss(y_train, model.predict(x_train)))
    err_test.append(loss(y_test, model.predict(x_test)))

> **Pause the video and make a prediction:**
>
> 1. What do you expect the training and test errors to look like when $K=1$?
> 2. As $K$ increases, how do you expect these two errors to change?

In [ ]:
# The test curve is shown here only to illustrate generalization.
# We should not use it to select K in practice.
plt.plot(K_list, err_test, '-', label='test')
plt.plot(K_list, err_train, '-', label='train')
plt.legend()
plt.xlabel('K (number of neighbours)')
plt.ylabel('mean squared error')
plt.show()

<div style="color: red;">
As you can see in the plot, when `K` is small, the training error is very low, while the test error, which is on the unseen test data is pretty high. As `K` increases, the gap between the training and test errors becomes smaller. 

So why does this happen?
<br><br>
With a small value of `K`, each prediction is based on only a few training samples, so each individual sample has a strong influence on the prediction. The model follows local details in the training data very closely, including noise. This gives us a low training error, but it can cause poor performance on test data. This is what we call overfitting.
<br><br>
As we increase `K`, each prediction is based on more neighbours, so the model becomes less sensitive to local details and noise. So the test error initially decreases. But, if `K` becomes too large, the model starts to average over too many samples, including samples that may be far away. For example, when predicting house prices downtown, a very large `K` might include houses from the suburbs as well. Then model misses important local patterns, and the test error starts to rise again. This phenomenon is called underfitting.
</div>

| Value of $K$ | Training error | Test error | Model behaviour |
|---|---:|---:|---|
| Small $K$ | Very low | High | **Overfitting** |
| Moderate $K$ | Slightly higher | Lowest | **Best generalization** |
| Very large $K$ | High | High | **Underfitting** |

> **Pause the video and think about this question:**
>
> If we select the best $K$ using training error, will it minimize the test error as well?

<div style="color: red;">
From this plot, we can see that the value of `K` with the lowest training error is `K=1`. But, the lowest test error occurs around `K=8`. So the hyperparameter that we found on the training set is not necessarily the one that performs best on test data. 
</div>


We care about generalization because we do not want a model that underfits or overfits. We want to find a model that performs well on unseen data—in other words, a model that generalizes well.

## 3. How Do We Select a Model That Generalizes Well?

### 3.1 Why Do We Need a Validation Set?

<div style="color: red;">
Can we use the training set?

The answer is: We should never choose hyperparameters by looking at the training error. Because our purpose is to train the model that performs well on unseen data, but the model has already seen every sample in the training set. Another reason is the lowest training error does not imply the lowest generalization error, as we've already discussed in the previous section. So training error is not suitable for model selection or hyperparameter selection.
</div>

- Why Can't We Use the Training Set?
    - Our goal is good performance on unseen data.
    - The lowest training error does not imply the lowest generalization error.

<div style="color: red;">
How about the test set?

We should not use that either. Because there will be test leakage problem. If we select a model based on test error, we'll make the test set part of the model selection process, which is something that we should avoid in our research. In fact, the test set should only be reserved for final evaluation.
</div>

- Why Can't We Use the Test Set?
    - **Test leakage problem:** Selecting a model based on test error makes the test set part of the model-selection process.
    - The test set should be reserved for final evaluation.

<div style="color: red;">
The standard solution is to introduce a validation set and use it to select the model or tune the hyperparameters. Overall, we divide the data into three parts: a training set, a validation set, and a test set. The training set is used to train the model, the validation set is used to select the model and tune the hyperparameters, and the test set is used for the final evaluation. We should never use the test set to choose the model or its hyperparameters, because that would introduce test leakage.
</div>

- The Role of Each Dataset
    - **Training set:** Train the model.
    - **Validation set:** Select the model and hyperparameters.
    - **Test set:** Final evaluation.

![Training, validation, and test data split](https://raw.githubusercontent.com/mikemikezhu/COMP551_McGill/main/assets/data-split.svg)

### 3.2 How Do We Select a Hyperparameter Using a Validation Set?

1. First, split the data into training, validation, and test sets.
2. Loop over candidate values of the hyperparameter.
3. Train each model using the training set.
4. Compute the validation error.
5. After the loop, select the hyperparameter with the lowest validation error.
6. Combine the training and validation sets.
7. Retrain the model using the selected hyperparameter.
8. Evaluate the final model on the test set.

<div style="color: red;">
Now let's have a look how to implement this procedure.
<br><br>
First, we determine the sizes of the test and validation sets. Here, we assign 10% of the data to each one. 

Next, we generate a random permutation of the indices to shuffle the data, so the split is not affected by the original ordering of the samples. 

We then create the training, validation, and test sets.
<br><br>
Next, we loop over the candidate values of `K`. For each value, we train the KNN model using only the training set and then we compute the training and validation errors.
<br><br>
Then we plot the validation error for each value of `K`. We select the value with the lowest validation error as the best `K`. Once `K` has been selected, we combine the training and validation sets and retrain the final model using all of that data. Finally, we evaluate the model on the test set to estimate its generalization performance.
</div>

In [ ]:
# Split the data into train (80%), validation (10%), and test (10%) sets
n_test = num_instances // 10
n_valid = num_instances // 10

inds = np.random.permutation(num_instances)

x_test, y_test = x[inds[:n_test]], y[inds[:n_test]]
x_valid, y_valid = x[inds[n_test:n_test + n_valid]], y[inds[n_test:n_test + n_valid]]
x_train, y_train = x[inds[n_test + n_valid:]], y[inds[n_test + n_valid:]]

# Select K using validation error only
K_list = range(1, 30)
err_valid = []

for K in K_list:
    model = neighbors.KNeighborsRegressor(n_neighbors=K)
    model.fit(x_train, y_train)
    err_valid.append(loss(y_valid, model.predict(x_valid)))

plt.plot(K_list, err_valid, label='validation')
plt.legend()
plt.xlabel('K (number of neighbours)')
plt.ylabel('mean squared error')
plt.show()

best_index = np.argmin(err_valid)
best_K = K_list[best_index]
print(f'Best K: {best_K}')
print(f'Validation error: {err_valid[best_index]:.4f}')

# Retrain on all non-test data
x_train_final = np.concatenate([x_train, x_valid], axis=0)
y_train_final = np.concatenate([y_train, y_valid], axis=0)

final_model = neighbors.KNeighborsRegressor(n_neighbors=best_K)
final_model.fit(x_train_final, y_train_final)

# Evaluate on the test set once, after model selection
test_error = loss(y_test, final_model.predict(x_test))
print(f'Final test error: {test_error:.4f}')

### 3.3 Why Do We Need Cross-Validation?

<div style="color: red;">
We have just seen how to use a validation set to select a model or tune its 
</div>

- A single validation split can be sensitive to the particular random split.
- Repeating training and validation across multiple folds gives us a more reliable estimate of validation performance.
- Cross-validation reduces our dependence on one particular validation split.

### 3.4 How Do We Perform Cross-Validation?

<div style="color: red;">
Here is how it works. We first divide the full dataset into two parts: 10% is the test set, and the remaining 90% is used for training and validation. We then divide that 90% into `N` folds. For example, in 5-fold cross-validation, we create five parts.
<br><br>
In the first round, the first fold is used for validation and the other four folds are used for training. In the second round, the second fold becomes the validation set and the remaining folds are used for training. We continue until each fold has served as the validation set once. This gives us five validation errors, which we can then average. The average is less dependent on a single random split and gives us a more stable estimate of validation performance.
</div>

![Five-fold cross-validation with test set](https://raw.githubusercontent.com/mikemikezhu/COMP551_McGill/main/assets/five-fold-cross-validation.svg)

<div style="color: red;">
Now let's have a look at how to implement cross-validation in python.
<br><br>
We begin by defining a `cross_validate` function. Suppose we have `n` samples and want to create five folds. The size of each validation fold is `n` divided by the number of folds.
<br><br>
We then loop over the folds, using `f` as the fold index. For each fold, we define the validation indices. For example, if `n_val = 20`, fold 2 uses the indices in the interval `[40, 60)`. The training indices consist of the samples before the validation fold and the samples after it. Finally, the function returns the training and validation indices.
</div>

In [ ]:
def cross_validate(n, n_folds=5):
    """
    Generator function for K-fold cross-validation splits.
    
    Divides n samples into n_folds non-overlapping folds. In each iteration,
    one fold is used for validation and the remaining folds for training.
    
    Parameters:
    n: total number of samples to split
    n_folds: number of folds (default 5)
    
    Yields:
    tr_inds: list of training indices
    val_inds: list of validation indices
    
    Example with n=100, n_folds=5:
    - Fold 0: train on [20-99], validate on [0-19]
    - Fold 1: train on [0-19, 40-99], validate on [20-39]
    - ...
    - Fold 4: train on [0-79], validate on [80-99]
    """
    # Calculate validation set size for each fold
    # Integer division ensures even split (last fold may be slightly larger)
    n_val = n // n_folds
    
    # Note: This line creates empty list (overwriting unused permutation)
    # The original line 'inds = np.random.permutation(n)' is not used
    inds = []
    
    # Iterate through each fold
    for f in range(n_folds):
        # Initialize empty training indices list
        tr_inds = []
        
        # Calculate validation indices for this fold
        # Fold f uses indices from [f*n_val, (f+1)*n_val)
        # Example: fold 2 with n_val=20 uses indices [40, 60)
        val_inds = list(range(f * n_val, (f+1)*n_val))
        
        # Build training indices from all OTHER folds
        # Training set = all indices except current validation fold
        
        # Add indices BEFORE current fold (folds 0 to f-1)
        if f > 0:
            # list(range(f*n_val)) creates [0, 1, ..., f*n_val-1]
            # These are all indices before the current validation fold
            tr_inds = list(range(f*n_val))
        
        # Add indices AFTER current fold (folds f+1 to n_folds-1)
        if f < n_folds - 1:
            # list(range((f+1)*n_val, n)) creates [(f+1)*n_val, ..., n-1]
            # Concatenate with existing tr_inds using + operator
            # These are all indices after the current validation fold
            tr_inds = tr_inds + list(range((f+1)*n_val, n))
        
        # The yield statement makes this a generator function
        # It suspends execution and returns (tr_inds, val_inds)
        # Next iteration resumes from here with the next fold
        # This is memory efficient: doesn't create all splits at once
        yield tr_inds, val_inds

<div style="color: red;">
After defining the cross-validation function, we need to first split the test data. Everything that remains is stored in `x_rest` and `y_rest`, and this is the data we are gonna use for cross-validation.
</div>

In [ ]:
# Create random permutation of all data indices
# This shuffles the data before splitting
# inds shape: (num_instances,)
inds = np.random.permutation(num_instances)

# Separate the test data (first n_test samples after shuffling)
# Advanced indexing: inds[:n_test] selects first n_test shuffled indices
# x_test shape: (n_test, num_features)
# y_test shape: (n_test,)
x_test, y_test = x[inds[:n_test]], y[inds[:n_test]]

# Keep remaining data for training and cross-validation
# inds[n_test:] selects all indices after n_test
# x_rest shape: (num_instances - n_test, num_features)
# y_rest shape: (num_instances - n_test,)
x_rest, y_rest = x[inds[n_test:]], y[inds[n_test:]]

# Calculate size of remaining data (for cross-validation)
# This will be split into n_folds during cross-validation
n_rest = num_instances - n_test

<div style="color: red;">
Next, we use cross-validation to select the best hyperparameter.
<br><br>
We first loop over the candidate values of `K`. Inside that loop, we call the cross-validation function to obtain the training and validation indices for each fold. We train the model on the training fold and compute the validation loss on the validation fold.
<br><br>
Once cross-validation is complete, we average the validation losses across folds for each value of `K`. We then select the `K` with the lowest mean validation loss. Finally, we retrain the model using the selected value of `K` and evaluate that final model on the test set.
</div>

In [ ]:
# Perform cross-validation to find the best K with error estimates
num_folds = 5

# Store one validation error per K per fold
# Shape: (number of candidate K values, number of folds)
err_valid = np.zeros((len(K_list), num_folds))

# Outer loop: iterate over different K values (hyperparameter search)
for i, K in enumerate(K_list):
    # Inner loop: iterate over cross-validation folds
    # enumerate() gives both fold index (f) and the split (tr, val)
    # cross_validate() is a generator that yields training/validation indices
    for f, (tr, val) in enumerate(cross_validate(n_rest, num_folds)):
        # Create KNN model with current K
        model = neighbors.KNeighborsRegressor(n_neighbors=K)
        
        # Train on training fold using advanced indexing
        # x_rest[tr] selects rows at indices in tr list
        # tr is a list of indices, x_rest[tr] selects those rows
        # Shape: x_rest[tr] is (len(tr), num_features)
        model = model.fit(x_rest[tr], y_rest[tr])
        
        # Evaluate on validation fold
        # val is a list of indices for validation samples
        # x_rest[val] selects validation samples, shape (len(val), num_features)
        # Store error in 2D array: row i is K value, column f is fold number
        err_valid[i, f] = loss(y_rest[val], model.predict(x_rest[val]))

# Average the validation errors across folds for each K
mean_valid_error = np.mean(err_valid, axis=1)

# Select the K with the lowest mean validation error
best_index = np.argmin(mean_valid_error)
best_K = K_list[best_index]
print(f'Best K: {best_K}')
print(f'Mean cross-validation error: {mean_valid_error[best_index]:.4f}')

# Refit one final model on all non-test data using the selected K
final_model = neighbors.KNeighborsRegressor(n_neighbors=best_K)
final_model.fit(x_rest, y_rest)

# Evaluate on the test set only once, after model selection is complete
test_error = loss(y_test, final_model.predict(x_test))
print(f'Final test error: {test_error:.4f}')

## 4. How Do We Evaluate the Final Model?

<div style="color: red;">
Finally, let's discuss evaluation metrics. For classification tasks, we can evaluate a model using metrics such as cross-entropy loss or accuracy. These metrics summarize the model's overall performance, but what if we wanna know which classes the model tends to confuse with one another?
<br><br>
So we can use a confusion matrix. In binary classification, the confusion matrix is a 2 by 2 matrix containing the numbers of true negatives, false positives, false negatives, and true positives. We can also use these values to calculate metrics such as precision and recall.
<br><br>
Now let's have a look at how to implement a confusion matrix in python.
</div>

So far, we have used a regression example to study generalization and model selection. Now, let's switch to a classification problem and look at a useful tool for understanding a classifier's errors: the confusion matrix.

A confusion matrix shows how predictions compare with the true labels. Rows represent the true classes, columns represent the predicted classes, diagonal entries are correct predictions, and off-diagonal entries are classification errors.

<div style="color: red;">
First, we load the Iris dataset, which is a classification dataset. We split it into training and test sets, train a decision tree on the training set, and then use it to make predictions on the test set. We store those predictions in `y_hat_test`.
</div>

In [ ]:
from sklearn import tree, model_selection

# Load the Iris classification dataset
x, y = datasets.load_iris(return_X_y=True)

# Split into train (80%) and test (20%) sets
x_train, x_test, y_train, y_test = model_selection.train_test_split(
    x, y, test_size=0.2, random_state=1234
)

# Train a decision tree and predict the test labels
yh_test = tree.DecisionTreeClassifier(random_state=1234).fit(
    x_train, y_train
).predict(x_test)

<div style="color: red;">
Now let's focus on how to implement the confusion matrix.
<br><br>
We first initialize a matrix with dimensions `n_classes` by `n_classes`. For example, in binary classification, this would be a 2 by 2 matrix.
<br><br>
Then we use two loops over the classes. In the outer loop, `c1` represents the ground-truth class. In the inner loop, `c2` represents the predicted class. Together, they define a `(true_class, predicted_class)` pair.
<br><br>
Inside the loop, `(y == c1)` produces a Boolean array indicating which samples truly belong to class `c1`. Similarly, `(yh == c2)` indicates which samples were predicted as class `c2`. We combine the two conditions using element-wise multiplication, which acts like an AND operation here: the result is 1 when both conditions are true and 0 otherwise.
<br><br>
Finally, we use `numpy sum` to count how many samples satisfy both conditions. This tells us how many samples truly belong to class `c1` but were predicted as class `c2`. Then we store that count in the corresponding entry of the confusion matrix. We can also print the confusion matrix we've got.
</div>

In [ ]:
def confusion_matrix(y, yh):
    """
    Compute confusion matrix for multi-class classification.
    
    The confusion matrix shows how predictions compare to true labels.
    Rows represent true classes, columns represent predicted classes.
    Element [i,j] = number of samples with true class i predicted as class j.
    
    Parameters:
    y: true class labels, shape (N,)
    yh: predicted class labels, shape (N,)
    
    Returns:
    c_matrix: confusion matrix, shape (n_classes, n_classes)
    
    Example for binary classification:
                 Predicted
                 0    1
    True  0  [[TN   FP]
          1   [FN   TP]]
    """
    # Calculate number of classes
    # np.max(y) + 1 assumes classes are labeled 0, 1, ..., K-1
    n_classes = np.max(y) + 1
    
    # Initialize confusion matrix with zeros
    # Shape: (n_classes, n_classes)
    c_matrix = np.zeros((n_classes, n_classes))
    
    # Nested loops to count predictions for each (true_class, predicted_class) pair
    for c1 in range(n_classes):
        for c2 in range(n_classes):
            # Boolean arithmetic to count matching conditions
            # (y==c1) creates boolean array: True where true label is c1, shape (N,)
            # (yh==c2) creates boolean array: True where prediction is c2, shape (N,)
            # * operator: element-wise multiplication (AND operation)
            #   Result is 1 when both conditions true, 0 otherwise
            # np.sum() counts total number of True values (sum of 1s)
            # This counts: samples with true class c1 predicted as c2
            c_matrix[c1, c2] = np.sum((y==c1)*(yh==c2))
    
    return c_matrix

# Compute and display confusion matrix
cmat = confusion_matrix(y_test, yh_test)
print(cmat)

For binary classification, the four entries of the confusion matrix are:

- True positive (TP)
- True negative (TN)
- False positive (FP)
- False negative (FN)

Metrics such as accuracy, precision, recall, false-positive rate, and false-discovery rate can be calculated from these values.

## 5. Summary
- Generalization: a model's capability to perform well on unseen samples sampled from the same underlying data distribution.
- The role of each dataset
    - **Training set:** Train the model.
    - **Validation set:** Select the model and hyperparameters.
    - **Test set:** Final evaluation. (Should NOT used for model selection)
- How to select a hyperparameter using validation set.
- Cross-validation
- Confusion matrix

<div style="color: red;">
That’s all for today’s code review session. If you have any questions, feel free to post them on Ed or come to my office hours. Thanks for watching, and see you next time!
</div>